In [14]:
import carla
import time
import pygame
import logging
import cv2
import numpy as np

In [15]:
CARLA_LOCALHOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 20.0

IMAGE_Z = 3
IMAGE_X = 0

logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
    )

logger = logging.getLogger(__name__)

camera_init_trans = carla.Transform(carla.Location(z = IMAGE_Z, x = IMAGE_X))

In [16]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_LOCALHOST, CARLA_PORT)
    client.set_timeout(20)
    world = client.get_world()

    return client, world

In [17]:
def setup(world: carla.World) -> None:

    logging.basicConfig(
        level = logging.INFO,
        format ="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
    )

    logger = logging.getLogger(__name__)
    
    try:
        for actor in world.get_actor().filter("vehicle.*"):
            actor.destroy()
    except:
        logger.info("No vehicle found, during setup.")
    finally:
        logger.info("Scene is clear.")



In [18]:
def spawn_vehicle(world: carla.World) -> carla.Vehicle:

    actors = world.get_actors().filter("vehicle.*")

    for vehicle in actors:
        vehicle.destroy()

    vehicle_bp = world.get_blueprint_library().filter("*cybertruck*")

    if not vehicle_bp:
        print("No Cybertrunk blueprint was found !")
        return None
    
    spawn_points = world.get_map().get_spawn_points()

    vehicle = world.try_spawn_actor(
        vehicle_bp[0], 
        spawn_points[0]
    )

    if vehicle is None:
        print("Failed to spawn vehicle !")
        return None

    return vehicle

In [19]:
def spawn_rgb_camera(world: carla.World, vehicle: carla.Vehicle) -> tuple[carla.Sensor, int, int]:

    rbg_bp = world.get_blueprint_library().find("sensor.camera.rgb")
    image_w = rbg_bp.get_attribute("image_size_x").as_int()
    image_h = rbg_bp.get_attribute("image_size_y").as_int()

    if not rbg_bp:
        raise RuntimeError(
            "No valid RGB blueprint found."
        )

    rgb=world.spawn_actor(rbg_bp, camera_init_trans, attach_to=vehicle)

    
    return rgb, image_w, image_h

In [36]:
def control_vehicle(vehicle, keys):

    control = carla.VehicleControl()

    # Throttle and brake
    if keys[pygame.K_w]:
        control.throttle = 1

    if keys[pygame.K_s]:
        control.brake = 0.1

    # Steering
    if keys[pygame.K_a]:
        control.steer = -0.5
    elif keys[pygame.K_d]:
        control.steer = 0.5

    vehicle.apply_control(control)

In [21]:
def set_weather_conditions(world):
    world.set_weather(carla.WeatherParameters.ClearNight)

In [22]:
def end_world(world, vehicle):

    vehicles = world.get_actors().filter("vehicle.*")
    for vehicle in vehicles:
        vehicle.destroy()

    print(f"Destroyed {len(vehicles)} vehicles.")

In [32]:
def follow_vehicle(world, vehicle):

    spectator = world.get_spectator()

    spectator.set_transform(
        carla.Transform(
            carla.Location(
                x=0,
                y=0,
                z=200
            ),
            carla.Rotation(
                pitch=-90,
                yaw=0,
                roll=0
            )
        )
    )

In [24]:
def rgb_callback(data, data_dict):
    data_dict["rgb_data"] = np.reshape(np.copy(data.raw_data), (data.height, data.width, 4))

In [37]:
def main():

    client, world = connect_to_carla()
    setup(world)

    vehicle = spawn_vehicle(world)

    rgb, image_w, image_h = spawn_rgb_camera(world, vehicle)

    data_dict = {
        "rgb_data": np.zeros((image_h, image_w), dtype=np.uint8)
    }

    rgb.listen(lambda image: rgb_callback(image, data_dict))

    # Initialize Pygame
    pygame.init()

    screen = pygame.display.set_mode((400, 200))
    pygame.display.set_caption("CARLA Manual Control")

    clock = pygame.time.Clock()

    running = True
    frame = 0
    try:
        while running:
            follow_vehicle(world, vehicle)
            world.tick()
            if frame % 2 == 0:
                follow_vehicle(world, vehicle)
            frame += 1
            # Check Pygame events
            for event in pygame.event.get():

                if event.type == pygame.QUIT:
                    running = False

                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_q:
                        running = False

            # Read keyboard state
            keys = pygame.key.get_pressed()

            # Apply vehicle control
            control_vehicle(vehicle, keys)

            # Display camera
            cv2.imshow("CARLA Grayscale Camera", data_dict["rgb_data"])

            # Check OpenCV keyboard events
            if cv2.waitKey(1) & 0xFF == ord('q'):
                running = False

            clock.tick(60)

    finally:
        vehicle.apply_control(carla.VehicleControl())

        rgb.stop()
        rgb.destroy()
        vehicle.destroy()

        pygame.quit()
        cv2.destroyAllWindows()

In [38]:
if __name__ == "__main__":
    main()
    

2026-11-13 22:36:26,245 [INFO] __main__: No vehicle found, during setup.
2026-11-13 22:36:26,247 [INFO] __main__: Scene is clear.
